In [1]:
#@title Cell 1 - Install
%pip -q install ultralytics==8.4.102 gdown opencv-python-headless pycocotools pandas pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 70.3 MB/s eta 0:00:00


In [2]:
#@title Cell 2 - Download official AIMECORAL repository to temporary Colab storage

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from typing import Any, Dict, List, Optional
from contextlib import redirect_stdout
from io import StringIO
import json
import hashlib
import shutil
import subprocess
import re
import os

import numpy as np
import pandas as pd

OFFICIAL_REPOSITORY = (
    "https://github.com/Youassin/AIMECORAL.git"
)

DATASET_VARIANT = "AIMECORAL1"

# Set these only when an annotation source is supplied separately.
MANUAL_IMAGE_ROOT = ""
MANUAL_ANNOTATION_PATH = ""

V3_TAG = "multiseed_wbf_results_ready_v3_complete"
CORAL_ONLY_TAG = (
    "multiseed_wbf_results_ready_v3_"
    "extension_yolov10n_coral_only_v1"
)
OUTPUT_TAG = (
    "multiseed_wbf_results_ready_v3_"
    "extension_final_combos"
)

SEED = 42
PROTOCOLS_TO_EVALUATE = [
    "time_based",
    "source_based",
]

CHECKPOINT_SOURCE = "coral_only"
IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}

ROOT_CANDIDATES = [
    Path("/content/drive/MyDrive/au-fish"),
    Path(
        "/content/drive/.shortcut-targets-by-id/"
        "1kW1zeaSZ1KWylT60LjX9VMVZYNBF137Z/au-fish"
    ),
]

def resolve_project_root() -> Path:
    for candidate in ROOT_CANDIDATES:
        if (candidate / "dataset_v2").is_dir():
            return candidate.resolve()

    for root in [
        Path("/content/drive/MyDrive"),
        Path("/content/drive/.shortcut-targets-by-id"),
    ]:
        if not root.exists():
            continue

        for dataset_root in root.rglob("dataset_v2"):
            if dataset_root.is_dir():
                return dataset_root.parent.resolve()

    raise FileNotFoundError(
        "Folder AU-FISH tidak ditemukan."
    )

PROJECT_ROOT = resolve_project_root()
DATASET_ROOT = PROJECT_ROOT / "dataset_v2"

V3_ROOT = DATASET_ROOT / "experiments" / V3_TAG

CORAL_ONLY_ROOT = (
    DATASET_ROOT
    / "experiments"
    / CORAL_ONLY_TAG
)

OUTPUT_ROOT = (
    DATASET_ROOT
    / "experiments"
    / OUTPUT_TAG
)

EXTERNAL_ROOT = (
    OUTPUT_ROOT
    / "external_aimecoral"
)

REPORTS_ROOT = (
    OUTPUT_ROOT
    / "reports"
)

MANIFESTS_ROOT = (
    OUTPUT_ROOT
    / "manifests"
)

for folder in [
    EXTERNAL_ROOT,
    REPORTS_ROOT,
    MANIFESTS_ROOT,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )

LOCAL_REPOSITORY = Path(
    "/content/AIMECORAL"
)

if LOCAL_REPOSITORY.exists():
    shutil.rmtree(
        LOCAL_REPOSITORY
    )

clone_result = subprocess.run(
    [
        "git",
        "clone",
        "--depth",
        "1",
        OFFICIAL_REPOSITORY,
        str(
            LOCAL_REPOSITORY
        ),
    ],
    text=True,
    capture_output=True,
)

download_log = {
    "repository": OFFICIAL_REPOSITORY,
    "returncode": clone_result.returncode,
    "stdout": clone_result.stdout,
    "stderr": clone_result.stderr,
}

(
    REPORTS_ROOT
    / "aimecoral_download_log.txt"
).write_text(
    json.dumps(
        download_log,
        indent=2,
    ),
    encoding="utf-8",
)

if clone_result.returncode != 0:
    raise RuntimeError(
        "AIMECORAL repository clone failed. "
        "See aimecoral_download_log.txt."
    )

if MANUAL_IMAGE_ROOT:
    IMAGE_ROOT = Path(
        MANUAL_IMAGE_ROOT
    )
else:
    IMAGE_ROOT = (
        LOCAL_REPOSITORY
        / DATASET_VARIANT
    )

if not IMAGE_ROOT.exists():
    raise FileNotFoundError(
        IMAGE_ROOT
    )

print("Image root:", IMAGE_ROOT)
print("External output:", EXTERNAL_ROOT)

Mounted at /content/drive
Image root: /content/AIMECORAL/AIMECORAL1
External output: /content/drive/.shortcut-targets-by-id/1kW1zeaSZ1KWylT60LjX9VMVZYNBF137Z/au-fish/dataset_v2/experiments/multiseed_wbf_results_ready_v3_extension_final_combos/external_aimecoral


In [3]:
#@title Cell 3 - Discover or convert detection annotations to one-class COCO

from PIL import Image

def load_json(path: Path):
    with path.open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(handle)

def atomic_json(value: Any, path: Path):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary = path.with_suffix(
        path.suffix + ".tmp"
    )

    with temporary.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            value,
            handle,
            ensure_ascii=False,
            indent=2,
            default=str,
        )

    temporary.replace(path)

def save_csv(
    dataframe: pd.DataFrame,
    path: Path,
):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    dataframe.to_csv(
        path,
        index=False,
    )

def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()

image_paths = sorted([
    path
    for path in IMAGE_ROOT.rglob("*")
    if (
        path.is_file()
        and path.suffix.lower()
        in IMAGE_EXTENSIONS
    )
])

if not image_paths:
    raise RuntimeError(
        "No image files were found."
    )

basename_index = {}

for path in image_paths:
    basename_index.setdefault(
        path.name,
        [],
    ).append(path)

def resolve_by_name(file_name: str):
    candidate = (
        IMAGE_ROOT
        / str(file_name)
    )

    if candidate.exists():
        return candidate.resolve()

    matches = basename_index.get(
        Path(
            str(file_name)
        ).name,
        [],
    )

    if len(matches) == 1:
        return matches[0].resolve()

    return None

def convert_coco(source_path: Path):
    source = load_json(
        source_path
    )

    if not (
        isinstance(source, dict)
        and isinstance(
            source.get("images"),
            list,
        )
        and isinstance(
            source.get("annotations"),
            list,
        )
    ):
        return None

    source_images = {
        int(image["id"]):
        image
        for image
        in source["images"]
    }

    converted_images = []
    image_id_map = {}
    path_by_new_id = {}

    for new_id, (
        old_id,
        image,
    ) in enumerate(
        sorted(
            source_images.items()
        ),
        start=1,
    ):
        resolved = resolve_by_name(
            image.get(
                "file_name",
                "",
            )
        )

        if resolved is None:
            continue

        width = image.get("width")
        height = image.get("height")

        if not width or not height:
            with Image.open(
                resolved
            ) as opened:
                width, height = (
                    opened.size
                )

        image_id_map[old_id] = new_id
        path_by_new_id[new_id] = resolved

        converted_images.append({
            "id": new_id,
            "file_name": str(
                resolved
                .relative_to(
                    IMAGE_ROOT
                )
            ),
            "width": int(width),
            "height": int(height),
        })

    converted_annotations = []

    for annotation in source[
        "annotations"
    ]:
        old_image_id = int(
            annotation[
                "image_id"
            ]
        )

        if old_image_id not in image_id_map:
            continue

        bbox = annotation.get("bbox")

        if (
            not isinstance(
                bbox,
                list,
            )
            or len(bbox) != 4
        ):
            continue

        x, y, width, height = [
            float(value)
            for value in bbox
        ]

        if width <= 0 or height <= 0:
            continue

        converted_annotations.append({
            "id": len(
                converted_annotations
            )
            + 1,
            "image_id": (
                image_id_map[
                    old_image_id
                ]
            ),
            "category_id": 1,
            "bbox": [
                x,
                y,
                width,
                height,
            ],
            "area": float(
                width * height
            ),
            "iscrowd": int(
                annotation.get(
                    "iscrowd",
                    0,
                )
            ),
        })

    if not converted_annotations:
        return None

    return {
        "info": {
            "source_format": "COCO",
            "source_path": str(
                source_path
            ),
            "all_categories_mapped_to": (
                "coral"
            ),
        },
        "licenses": [],
        "images": converted_images,
        "annotations": (
            converted_annotations
        ),
        "categories": [
            {
                "id": 1,
                "name": "coral",
            }
        ],
    }

def convert_label_studio(
    source_path: Path,
):
    source = load_json(
        source_path
    )

    if not isinstance(
        source,
        list,
    ):
        return None

    images = []
    annotations = []
    image_key_to_id = {}

    for task in source:
        if not isinstance(
            task,
            dict,
        ):
            continue

        image_reference = (
            task.get(
                "data",
                {},
            ).get(
                "image"
            )
        )

        if not image_reference:
            continue

        resolved = resolve_by_name(
            str(
                image_reference
            ).split(
                "?"
            )[0]
        )

        if resolved is None:
            continue

        key = str(resolved)

        if key not in image_key_to_id:
            with Image.open(
                resolved
            ) as opened:
                width, height = (
                    opened.size
                )

            image_id = len(images) + 1
            image_key_to_id[key] = image_id

            images.append({
                "id": image_id,
                "file_name": str(
                    resolved.relative_to(
                        IMAGE_ROOT
                    )
                ),
                "width": int(width),
                "height": int(height),
            })

        image_id = image_key_to_id[key]

        for annotation_set in task.get(
            "annotations",
            [],
        ):
            for result in annotation_set.get(
                "result",
                [],
            ):
                value = result.get(
                    "value",
                    {},
                )

                if not all(
                    field in value
                    for field in [
                        "x",
                        "y",
                        "width",
                        "height",
                    ]
                ):
                    continue

                original_width = float(
                    result.get(
                        "original_width",
                        images[
                            image_id - 1
                        ][
                            "width"
                        ],
                    )
                )

                original_height = float(
                    result.get(
                        "original_height",
                        images[
                            image_id - 1
                        ][
                            "height"
                        ],
                    )
                )

                x = (
                    float(
                        value["x"]
                    )
                    / 100.0
                    * original_width
                )

                y = (
                    float(
                        value["y"]
                    )
                    / 100.0
                    * original_height
                )

                width = (
                    float(
                        value["width"]
                    )
                    / 100.0
                    * original_width
                )

                height = (
                    float(
                        value["height"]
                    )
                    / 100.0
                    * original_height
                )

                if width <= 0 or height <= 0:
                    continue

                annotations.append({
                    "id": len(
                        annotations
                    )
                    + 1,
                    "image_id": image_id,
                    "category_id": 1,
                    "bbox": [
                        x,
                        y,
                        width,
                        height,
                    ],
                    "area": float(
                        width * height
                    ),
                    "iscrowd": 0,
                })

    if not annotations:
        return None

    return {
        "info": {
            "source_format": (
                "Label Studio"
            ),
            "source_path": str(
                source_path
            ),
            "all_labels_mapped_to": (
                "coral"
            ),
        },
        "licenses": [],
        "images": images,
        "annotations": annotations,
        "categories": [
            {
                "id": 1,
                "name": "coral",
            }
        ],
    }

def convert_yolo_txt():
    label_candidates = [
        path
        for path in IMAGE_ROOT.rglob(
            "*.txt"
        )
        if path.name.lower()
        not in {
            "readme.txt",
            "license.txt",
        }
    ]

    if not label_candidates:
        return None

    images = []
    annotations = []

    for image_path in image_paths:
        likely_labels = [
            image_path.with_suffix(
                ".txt"
            ),
            Path(
                str(image_path)
                .replace(
                    "/images/",
                    "/labels/",
                )
            ).with_suffix(
                ".txt"
            ),
        ]

        label_path = next(
            (
                path
                for path
                in likely_labels
                if path.exists()
            ),
            None,
        )

        if label_path is None:
            continue

        with Image.open(
            image_path
        ) as opened:
            image_width, image_height = (
                opened.size
            )

        image_id = len(images) + 1

        images.append({
            "id": image_id,
            "file_name": str(
                image_path.relative_to(
                    IMAGE_ROOT
                )
            ),
            "width": int(
                image_width
            ),
            "height": int(
                image_height
            ),
        })

        for line in label_path.read_text(
            encoding="utf-8",
            errors="ignore",
        ).splitlines():
            parts = line.strip().split()

            if len(parts) < 5:
                continue

            _, center_x, center_y, width, height = [
                float(value)
                for value in parts[:5]
            ]

            width_pixels = (
                width * image_width
            )

            height_pixels = (
                height * image_height
            )

            x = (
                center_x * image_width
                - width_pixels / 2.0
            )

            y = (
                center_y * image_height
                - height_pixels / 2.0
            )

            if (
                width_pixels <= 0
                or height_pixels <= 0
            ):
                continue

            annotations.append({
                "id": len(
                    annotations
                )
                + 1,
                "image_id": image_id,
                "category_id": 1,
                "bbox": [
                    float(x),
                    float(y),
                    float(
                        width_pixels
                    ),
                    float(
                        height_pixels
                    ),
                ],
                "area": float(
                    width_pixels
                    * height_pixels
                ),
                "iscrowd": 0,
            })

    if not annotations:
        return None

    return {
        "info": {
            "source_format": "YOLO TXT",
            "all_classes_mapped_to": (
                "coral"
            ),
        },
        "licenses": [],
        "images": images,
        "annotations": annotations,
        "categories": [
            {
                "id": 1,
                "name": "coral",
            }
        ],
    }

converted = None
annotation_source = None

candidate_json_paths = []

if MANUAL_ANNOTATION_PATH:
    candidate_json_paths.append(
        Path(
            MANUAL_ANNOTATION_PATH
        )
    )
else:
    candidate_json_paths.extend(
        sorted(
            IMAGE_ROOT.rglob(
                "*.json"
            )
        )
    )

for candidate in candidate_json_paths:
    if not candidate.exists():
        continue

    try:
        converted = convert_coco(
            candidate
        )

        if converted is None:
            converted = (
                convert_label_studio(
                    candidate
                )
            )

        if converted is not None:
            annotation_source = (
                candidate
            )
            break

    except Exception as exception:
        print(
            "Annotation candidate skipped:",
            candidate,
            repr(exception),
        )

if converted is None:
    converted = convert_yolo_txt()

    if converted is not None:
        annotation_source = (
            "auto-discovered YOLO TXT"
        )

external_coco_path = (
    EXTERNAL_ROOT
    / DATASET_VARIANT
    / "annotations_coral_coco.json"
)

if converted is not None:
    atomic_json(
        converted,
        external_coco_path,
    )

    conversion_status = {
        "status": "READY_FOR_MAP",
        "dataset_variant": (
            DATASET_VARIANT
        ),
        "image_root": str(
            IMAGE_ROOT
        ),
        "images_found": len(
            image_paths
        ),
        "converted_images": len(
            converted["images"]
        ),
        "converted_annotations": len(
            converted[
                "annotations"
            ]
        ),
        "annotation_source": str(
            annotation_source
        ),
        "external_coco_path": str(
            external_coco_path
        ),
        "external_coco_sha256": (
            sha256_file(
                external_coco_path
            )
        ),
    }

else:
    conversion_status = {
        "status": (
            "DEFERRED_NO_BOUNDING_BOX_ANNOTATIONS"
        ),
        "dataset_variant": (
            DATASET_VARIANT
        ),
        "image_root": str(
            IMAGE_ROOT
        ),
        "images_found": len(
            image_paths
        ),
        "reason": (
            "The downloaded dataset contains images, "
            "but no supported COCO, Label Studio, or "
            "YOLO bounding-box annotation was found. "
            "mAP cannot be computed without ground truth."
        ),
        "manual_annotation_path": (
            MANUAL_ANNOTATION_PATH
        ),
    }

atomic_json(
    conversion_status,
    MANIFESTS_ROOT
    / "aimecoral_conversion_status.json",
)

print(json.dumps(conversion_status, indent=2))

{
  "status": "READY_FOR_MAP",
  "dataset_variant": "AIMECORAL1",
  "image_root": "/content/AIMECORAL/AIMECORAL1",
  "images_found": 400,
  "converted_images": 400,
  "converted_annotations": 4061,
  "annotation_source": "auto-discovered YOLO TXT",
  "external_coco_path": "/content/drive/.shortcut-targets-by-id/1kW1zeaSZ1KWylT60LjX9VMVZYNBF137Z/au-fish/dataset_v2/experiments/multiseed_wbf_results_ready_v3_extension_final_combos/external_aimecoral/AIMECORAL1/annotations_coral_coco.json",
  "external_coco_sha256": "0782e7b8a9cff19d4b067f45a2d2da33019b26cd8c65c4c564d82be285e2c904"
}


In [4]:
#@title Cell 4 - YOLOv10n inference on AIMECORAL when ground truth is available

import torch
from ultralytics import YOLO

def checkpoint_path(
    protocol: str,
) -> Path:
    if CHECKPOINT_SOURCE == "coral_only":
        return (
            CORAL_ONLY_ROOT
            / "runs"
            / protocol
            / "yolov10n"
            / f"seed_{SEED}"
            / "train"
            / "weights"
            / "best.pt"
        )

    raise ValueError(
        "Only coral_only is configured "
        "for this external coral evaluation."
    )

prediction_rows = []
inference_status_rows = []

if conversion_status[
    "status"
] == "READY_FOR_MAP":
    external_gt = load_json(
        external_coco_path
    )

    external_images = sorted(
        external_gt["images"],
        key=lambda item: int(
            item["id"]
        ),
    )

    external_paths = [
        (
            IMAGE_ROOT
            / image["file_name"]
        ).resolve()
        for image in external_images
    ]

    for protocol in (
        PROTOCOLS_TO_EVALUATE
    ):
        checkpoint = checkpoint_path(
            protocol
        )

        if not checkpoint.exists():
            inference_status_rows.append({
                "protocol": protocol,
                "status": (
                    "PENDING_CHECKPOINT"
                ),
                "checkpoint_path": str(
                    checkpoint
                ),
            })
            continue

        model = YOLO(
            str(checkpoint)
        )

        results = model.predict(
            source=[
                str(path)
                for path
                in external_paths
            ],
            stream=True,
            imgsz=640,
            conf=0.001,
            iou=0.70,
            max_det=300,
            augment=False,
            verbose=False,
            device=(
                0
                if torch.cuda.is_available()
                else "cpu"
            ),
        )

        predictions = []
        result_count = 0

        for result_count, result in enumerate(
            results,
            start=1,
        ):
            image_id = int(
                external_images[
                    result_count - 1
                ][
                    "id"
                ]
            )

            if result.boxes is None:
                continue

            boxes = (
                result.boxes.xyxy
                .detach()
                .cpu()
                .numpy()
            )

            scores = (
                result.boxes.conf
                .detach()
                .cpu()
                .numpy()
            )

            classes = (
                result.boxes.cls
                .detach()
                .cpu()
                .numpy()
                .astype(int)
            )

            for box, score, class_index in zip(
                boxes,
                scores,
                classes,
            ):
                class_name = str(
                    model.names[
                        int(
                            class_index
                        )
                    ]
                ).strip().lower()

                if class_name != "coral":
                    continue

                x1, y1, x2, y2 = [
                    float(value)
                    for value in box
                ]

                predictions.append({
                    "image_id": image_id,
                    "category_id": 1,
                    "bbox": [
                        x1,
                        y1,
                        x2 - x1,
                        y2 - y1,
                    ],
                    "score": float(
                        score
                    ),
                })

        if result_count != len(
            external_images
        ):
            raise RuntimeError(
                "Inference image count mismatch."
            )

        prediction_path = (
            EXTERNAL_ROOT
            / DATASET_VARIANT
            / protocol
            / (
                f"yolov10n_seed_{SEED}_"
                "predictions.json"
            )
        )

        atomic_json(
            predictions,
            prediction_path,
        )

        prediction_rows.append({
            "protocol": protocol,
            "checkpoint_path": str(
                checkpoint
            ),
            "prediction_path": str(
                prediction_path
            ),
            "detections": len(
                predictions
            ),
        })

        inference_status_rows.append({
            "protocol": protocol,
            "status": (
                "PREDICTIONS_READY"
            ),
            "checkpoint_path": str(
                checkpoint
            ),
            "prediction_path": str(
                prediction_path
            ),
        })

        del model

else:
    inference_status_rows.append({
        "protocol": None,
        "status": (
            "SKIPPED_NO_GROUND_TRUTH"
        ),
        "reason": (
            conversion_status[
                "reason"
            ]
        ),
    })

inference_status_df = pd.DataFrame(
    inference_status_rows
)

save_csv(
    inference_status_df,
    REPORTS_ROOT
    / "aimecoral_inference_status.csv",
)

display(inference_status_df)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


,protocol,status,checkpoint_path,prediction_path
0,time_based,PREDICTIONS_READY,/content/drive/.shortcut-targets-by-id/1kW1zea...,/content/drive/.shortcut-targets-by-id/1kW1zea...
1,source_based,PREDICTIONS_READY,/content/drive/.shortcut-targets-by-id/1kW1zea...,/content/drive/.shortcut-targets-by-id/1kW1zea...


In [5]:
#@title Cell 5 - COCO evaluation and AU-FISH comparison

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

def evaluate_external(
    gt_path: Path,
    prediction_path: Path,
):
    coco_gt = COCO(
        str(gt_path)
    )

    predictions = load_json(
        prediction_path
    )

    if not predictions:
        return {
            "map50_95": 0.0,
            "map50": 0.0,
            "map75": 0.0,
            "coral_ap": 0.0,
            "detections": 0,
        }

    coco_dt = coco_gt.loadRes(
        str(prediction_path)
    )

    evaluator = COCOeval(
        coco_gt,
        coco_dt,
        "bbox",
    )

    evaluator.params.imgIds = sorted(
        coco_gt.getImgIds()
    )

    evaluator.params.catIds = [1]
    evaluator.params.maxDets = [
        1,
        10,
        100,
    ]

    output_buffer = StringIO()

    with redirect_stdout(
        output_buffer
    ):
        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()

    precision = evaluator.eval[
        "precision"
    ][
        :,
        :,
        0,
        0,
        -1,
    ]

    valid = precision[
        precision > -1
    ]

    return {
        "map50_95": float(
            evaluator.stats[0]
        ),
        "map50": float(
            evaluator.stats[1]
        ),
        "map75": float(
            evaluator.stats[2]
        ),
        "coral_ap": (
            float(
                valid.mean()
            )
            if valid.size
            else float("nan")
        ),
        "detections": len(
            predictions
        ),
    }

evaluation_rows = []

if conversion_status[
    "status"
] == "READY_FOR_MAP":
    for row in inference_status_rows:
        if row.get(
            "status"
        ) != "PREDICTIONS_READY":
            continue

        protocol = row[
            "protocol"
        ]

        metrics = evaluate_external(
            external_coco_path,
            Path(
                row[
                    "prediction_path"
                ]
            ),
        )

        aufish_result_path = (
            CORAL_ONLY_ROOT
            / "reports"
            / "results_by_seed.csv"
        )

        aufish_coral_ap = float(
            "nan"
        )

        if aufish_result_path.exists():
            aufish_results = pd.read_csv(
                aufish_result_path
            )

            matched = aufish_results[
                (
                    aufish_results[
                        "protocol"
                    ]
                    == protocol
                )
                & (
                    aufish_results[
                        "seed"
                    ]
                    == SEED
                )
                & (
                    aufish_results[
                        "split"
                    ]
                    == "test"
                )
            ]

            if not matched.empty:
                aufish_coral_ap = float(
                    matched.iloc[0][
                        "coral_ap"
                    ]
                )

        evaluation_rows.append({
            "dataset": DATASET_VARIANT,
            "protocol_checkpoint": (
                protocol
            ),
            "model": (
                "yolov10n_coral_only"
            ),
            "seed": SEED,
            "status": "EVALUATED",
            **metrics,
            "aufish_coral_ap_seed_42": (
                aufish_coral_ap
            ),
            "external_minus_aufish_coral_ap": (
                metrics["coral_ap"]
                - aufish_coral_ap
                if np.isfinite(
                    aufish_coral_ap
                )
                else float("nan")
            ),
            "direct_dataset_equivalence": (
                False
            ),
        })

else:
    evaluation_rows.append({
        "dataset": DATASET_VARIANT,
        "protocol_checkpoint": None,
        "model": (
            "yolov10n_coral_only"
        ),
        "seed": SEED,
        "status": (
            "DEFERRED_NO_BOUNDING_BOX_ANNOTATIONS"
        ),
        "map50_95": float("nan"),
        "map50": float("nan"),
        "map75": float("nan"),
        "coral_ap": float("nan"),
        "detections": 0,
        "note": (
            conversion_status[
                "reason"
            ]
        ),
        "direct_dataset_equivalence": (
            False
        ),
    })

evaluation_df = pd.DataFrame(
    evaluation_rows
)

save_csv(
    evaluation_df,
    REPORTS_ROOT
    / "aimecoral_evaluation.csv",
)

atomic_json(
    {
        "official_repository": (
            OFFICIAL_REPOSITORY
        ),
        "dataset_variant": (
            DATASET_VARIANT
        ),
        "conversion_status": (
            conversion_status
        ),
        "evaluation_rows": len(
            evaluation_df
        ),
        "report_path": str(
            REPORTS_ROOT
            / "aimecoral_evaluation.csv"
        ),
        "interpretation": (
            "Cross-dataset results are contextual. "
            "They are not directly equivalent to "
            "AU-FISH because data collection, "
            "annotation, and split protocols differ."
        ),
    },
    MANIFESTS_ROOT
    / "aimecoral_completion.json",
)

display(evaluation_df)

loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.15s)
creating index...
index created!
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.04s)
creating index...
index created!


,dataset,protocol_checkpoint,model,seed,status,map50_95,map50,map75,coral_ap,detections,aufish_coral_ap_seed_42,external_minus_aufish_coral_ap,direct_dataset_equivalence
0,AIMECORAL1,time_based,yolov10n_coral_only,42,EVALUATED,0.009332,0.016848,0.007807,0.009332,3683,NaN,NaN,False
1,AIMECORAL1,source_based,yolov10n_coral_only,42,EVALUATED,0.002678,0.006628,0.001853,0.002678,8290,NaN,NaN,False


In [5]:
#@title Cell 5 - COCO evaluation and AU-FISH comparison

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

def evaluate_external(
    gt_path: Path,
    prediction_path: Path,
):
    coco_gt = COCO(
        str(gt_path)
    )

    predictions = load_json(
        prediction_path
    )

    if not predictions:
        return {
            "map50_95": 0.0,
            "map50": 0.0,
            "map75": 0.0,
            "coral_ap": 0.0,
            "detections": 0,
        }

    coco_dt = coco_gt.loadRes(
        str(prediction_path)
    )

    evaluator = COCOeval(
        coco_gt,
        coco_dt,
        "bbox",
    )

    evaluator.params.imgIds = sorted(
        coco_gt.getImgIds()
    )

    evaluator.params.catIds = [1]
    evaluator.params.maxDets = [
        1,
        10,
        100,
    ]

    output_buffer = StringIO()

    with redirect_stdout(
        output_buffer
    ):
        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()

    precision = evaluator.eval[
        "precision"
    ][
        :,
        :,
        0,
        0,
        -1,
    ]

    valid = precision[
        precision > -1
    ]

    return {
        "map50_95": float(
            evaluator.stats[0]
        ),
        "map50": float(
            evaluator.stats[1]
        ),
        "map75": float(
            evaluator.stats[2]
        ),
        "coral_ap": (
            float(
                valid.mean()
            )
            if valid.size
            else float("nan")
        ),
        "detections": len(
            predictions
        ),
    }

evaluation_rows = []

if conversion_status[
    "status"
] == "READY_FOR_MAP":
    for row in inference_status_rows:
        if row.get(
            "status"
        ) != "PREDICTIONS_READY":
            continue

        protocol = row[
            "protocol"
        ]

        metrics = evaluate_external(
            external_coco_path,
            Path(
                row[
                    "prediction_path"
                ]
            ),
        )

        aufish_result_path = (
            CORAL_ONLY_ROOT
            / "reports"
            / "results_by_seed.csv"
        )

        aufish_coral_ap = float(
            "nan"
        )

        if aufish_result_path.exists():
            aufish_results = pd.read_csv(
                aufish_result_path
            )

            matched = aufish_results[
                (
                    aufish_results[
                        "protocol"
                    ]
                    == protocol
                )
                & (
                    aufish_results[
                        "seed"
                    ]
                    == SEED
                )
                & (
                    aufish_results[
                        "split"
                    ]
                    == "test"
                )
            ]

            if not matched.empty:
                aufish_coral_ap = float(
                    matched.iloc[0][
                        "coral_ap"
                    ]
                )

        evaluation_rows.append({
            "dataset": DATASET_VARIANT,
            "protocol_checkpoint": (
                protocol
            ),
            "model": (
                "yolov10n_coral_only"
            ),
            "seed": SEED,
            "status": "EVALUATED",
            **metrics,
            "aufish_coral_ap_seed_42": (
                aufish_coral_ap
            ),
            "external_minus_aufish_coral_ap": (
                metrics["coral_ap"]
                - aufish_coral_ap
                if np.isfinite(
                    aufish_coral_ap
                )
                else float("nan")
            ),
            "direct_dataset_equivalence": (
                False
            ),
        })

else:
    evaluation_rows.append({
        "dataset": DATASET_VARIANT,
        "protocol_checkpoint": None,
        "model": (
            "yolov10n_coral_only"
        ),
        "seed": SEED,
        "status": (
            "DEFERRED_NO_BOUNDING_BOX_ANNOTATIONS"
        ),
        "map50_95": float("nan"),
        "map50": float("nan"),
        "map75": float("nan"),
        "coral_ap": float("nan"),
        "detections": 0,
        "note": (
            conversion_status[
                "reason"
            ]
        ),
        "direct_dataset_equivalence": (
            False
        ),
    })

evaluation_df = pd.DataFrame(
    evaluation_rows
)

save_csv(
    evaluation_df,
    REPORTS_ROOT
    / "aimecoral_evaluation.csv",
)

atomic_json(
    {
        "official_repository": (
            OFFICIAL_REPOSITORY
        ),
        "dataset_variant": (
            DATASET_VARIANT
        ),
        "conversion_status": (
            conversion_status
        ),
        "evaluation_rows": len(
            evaluation_df
        ),
        "report_path": str(
            REPORTS_ROOT
            / "aimecoral_evaluation.csv"
        ),
        "interpretation": (
            "Cross-dataset results are contextual. "
            "They are not directly equivalent to "
            "AU-FISH because data collection, "
            "annotation, and split protocols differ."
        ),
    },
    MANIFESTS_ROOT
    / "aimecoral_completion.json",
)

display(evaluation_df)

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.04s)
creating index...
index created!
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.07s)
creating index...
index created!


,dataset,protocol_checkpoint,model,seed,status,map50_95,map50,map75,coral_ap,detections,aufish_coral_ap_seed_42,external_minus_aufish_coral_ap,direct_dataset_equivalence
0,AIMECORAL1,time_based,yolov10n_coral_only,123,EVALUATED,0.015175,0.027242,0.013995,0.015175,9118,0.495750,-0.480575,False
1,AIMECORAL1,source_based,yolov10n_coral_only,123,EVALUATED,0.013611,0.033284,0.009251,0.013611,13819,0.610368,-0.596757,False


In [7]:
#@title Cell 5 - COCO evaluation and AU-FISH comparison

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

def evaluate_external(
    gt_path: Path,
    prediction_path: Path,
):
    coco_gt = COCO(
        str(gt_path)
    )

    predictions = load_json(
        prediction_path
    )

    if not predictions:
        return {
            "map50_95": 0.0,
            "map50": 0.0,
            "map75": 0.0,
            "coral_ap": 0.0,
            "detections": 0,
        }

    coco_dt = coco_gt.loadRes(
        str(prediction_path)
    )

    evaluator = COCOeval(
        coco_gt,
        coco_dt,
        "bbox",
    )

    evaluator.params.imgIds = sorted(
        coco_gt.getImgIds()
    )

    evaluator.params.catIds = [1]
    evaluator.params.maxDets = [
        1,
        10,
        100,
    ]

    output_buffer = StringIO()

    with redirect_stdout(
        output_buffer
    ):
        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()

    precision = evaluator.eval[
        "precision"
    ][
        :,
        :,
        0,
        0,
        -1,
    ]

    valid = precision[
        precision > -1
    ]

    return {
        "map50_95": float(
            evaluator.stats[0]
        ),
        "map50": float(
            evaluator.stats[1]
        ),
        "map75": float(
            evaluator.stats[2]
        ),
        "coral_ap": (
            float(
                valid.mean()
            )
            if valid.size
            else float("nan")
        ),
        "detections": len(
            predictions
        ),
    }

evaluation_rows = []

if conversion_status[
    "status"
] == "READY_FOR_MAP":
    for row in inference_status_rows:
        if row.get(
            "status"
        ) != "PREDICTIONS_READY":
            continue

        protocol = row[
            "protocol"
        ]

        metrics = evaluate_external(
            external_coco_path,
            Path(
                row[
                    "prediction_path"
                ]
            ),
        )

        aufish_result_path = (
            CORAL_ONLY_ROOT
            / "reports"
            / "results_by_seed.csv"
        )

        aufish_coral_ap = float(
            "nan"
        )

        if aufish_result_path.exists():
            aufish_results = pd.read_csv(
                aufish_result_path
            )

            matched = aufish_results[
                (
                    aufish_results[
                        "protocol"
                    ]
                    == protocol
                )
                & (
                    aufish_results[
                        "seed"
                    ]
                    == SEED
                )
                & (
                    aufish_results[
                        "split"
                    ]
                    == "test"
                )
            ]

            if not matched.empty:
                aufish_coral_ap = float(
                    matched.iloc[0][
                        "coral_ap"
                    ]
                )

        evaluation_rows.append({
            "dataset": DATASET_VARIANT,
            "protocol_checkpoint": (
                protocol
            ),
            "model": (
                "yolov10n_coral_only"
            ),
            "seed": SEED,
            "status": "EVALUATED",
            **metrics,
            "aufish_coral_ap_seed_42": (
                aufish_coral_ap
            ),
            "external_minus_aufish_coral_ap": (
                metrics["coral_ap"]
                - aufish_coral_ap
                if np.isfinite(
                    aufish_coral_ap
                )
                else float("nan")
            ),
            "direct_dataset_equivalence": (
                False
            ),
        })

else:
    evaluation_rows.append({
        "dataset": DATASET_VARIANT,
        "protocol_checkpoint": None,
        "model": (
            "yolov10n_coral_only"
        ),
        "seed": SEED,
        "status": (
            "DEFERRED_NO_BOUNDING_BOX_ANNOTATIONS"
        ),
        "map50_95": float("nan"),
        "map50": float("nan"),
        "map75": float("nan"),
        "coral_ap": float("nan"),
        "detections": 0,
        "note": (
            conversion_status[
                "reason"
            ]
        ),
        "direct_dataset_equivalence": (
            False
        ),
    })

evaluation_df = pd.DataFrame(
    evaluation_rows
)

save_csv(
    evaluation_df,
    REPORTS_ROOT
    / "aimecoral_evaluation.csv",
)

atomic_json(
    {
        "official_repository": (
            OFFICIAL_REPOSITORY
        ),
        "dataset_variant": (
            DATASET_VARIANT
        ),
        "conversion_status": (
            conversion_status
        ),
        "evaluation_rows": len(
            evaluation_df
        ),
        "report_path": str(
            REPORTS_ROOT
            / "aimecoral_evaluation.csv"
        ),
        "interpretation": (
            "Cross-dataset results are contextual. "
            "They are not directly equivalent to "
            "AU-FISH because data collection, "
            "annotation, and split protocols differ."
        ),
    },
    MANIFESTS_ROOT
    / "aimecoral_completion.json",
)

display(evaluation_df)

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.05s)
creating index...
index created!
loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.07s)
creating index...
index created!


,dataset,protocol_checkpoint,model,seed,status,map50_95,map50,map75,coral_ap,detections,aufish_coral_ap_seed_42,external_minus_aufish_coral_ap,direct_dataset_equivalence
0,AIMECORAL1,time_based,yolov10n_coral_only,2026,EVALUATED,0.015175,0.027242,0.013995,0.015175,9118,0.481470,-0.466295,False
1,AIMECORAL1,source_based,yolov10n_coral_only,2026,EVALUATED,0.013611,0.033284,0.009251,0.013611,13819,0.622363,-0.608752,False
